# V3 Universal Football Model — Training & Calibration Loop

This notebook covers the final steps of the Machine Learning pipeline:
1. **The PyTorch Training Loop:** Feeding the continuous features and league embeddings into the network using `FocalLoss`.
2. **Multi-Class Isotonic Calibration:** Neural networks are notoriously uncalibrated (often overconfident). We fit three separate Isotonic Regressors (Away, Draw, Home) on a validation set to map raw softmax outputs to true historical probabilities, ensuring that when the model says "70%", it exactly means 70%.

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import brier_score_loss

# Set seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)


## 1. Re-define the Model & Loss (From Phase 2)
We redefine them here so this notebook is fully self-contained and runnable.

In [3]:
class UniversalFootballNet(nn.Module):
    def __init__(self, n_continuous=10, num_leagues=100, embed_dim=4, h1=64, h2=32, dropout=0.3):
        super().__init__()
        self.league_embed = nn.Embedding(num_embeddings=num_leagues, embedding_dim=embed_dim)
        input_dim = n_continuous + embed_dim
        
        self.fc1 = nn.Linear(input_dim, h1)
        self.bn1 = nn.BatchNorm1d(h1)
        self.fc2 = nn.Linear(h1, h2)
        self.bn2 = nn.BatchNorm1d(h2)
        self.head = nn.Linear(h2, 3)
        self.drop = nn.Dropout(dropout)
        self.act = nn.ReLU()
        
    def forward(self, x_cont, x_league):
        league_vec = self.league_embed(x_league)
        x = torch.cat([x_cont, league_vec], dim=1)
        x = self.drop(self.act(self.bn1(self.fc1(x))))
        x = self.drop(self.act(self.bn2(self.fc2(x))))
        return self.head(x)

class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha
        
    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none', weight=self.alpha)
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        return focal_loss.mean()


## 2. Mock Dataset Setup
We generate 5,000 synthetic rows of data to act as our training and validation sets.

In [4]:
class MockFootballDataset(Dataset):
    def __init__(self, num_samples=5000):
        # Continuous: min, home_xi, away_xi, days_h, days_a, gd, score_state, h_red, a_red, is_ko
        self.cont = torch.rand(num_samples, 10) * 10 
        self.league = torch.randint(0, 5, (num_samples,)) # 5 dummy leagues
        self.targets = torch.randint(0, 3, (num_samples,)) # 0: Away, 1: Draw, 2: Home
        
    def __len__(self):
        return len(self.targets)
    
    def __getitem__(self, idx):
        return self.cont[idx], self.league[idx], self.targets[idx]

# Create Train and Validation DataLoaders
train_data = MockFootballDataset(num_samples=4000)
val_data = MockFootballDataset(num_samples=1000)

train_loader = DataLoader(train_data, batch_size=128, shuffle=True)
val_loader = DataLoader(val_data, batch_size=128, shuffle=False)


## 3. The Training Loop
A standard PyTorch training loop using the AdamW optimizer.

In [5]:
model = UniversalFootballNet(n_continuous=10, num_leagues=5)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.005, weight_decay=1e-4)
loss_fn = FocalLoss(alpha=torch.tensor([1.0, 1.5, 1.0]), gamma=2.0) # Bonus weight to draws

epochs = 3
print("Starting Training Loop...")
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for batch_cont, batch_league, batch_targets in train_loader:
        optimizer.zero_grad()
        logits = model(batch_cont, batch_league)
        loss = loss_fn(logits, batch_targets)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    print(f"Epoch {epoch+1}/{epochs} | Training Focal Loss: {total_loss/len(train_loader):.4f}")


Starting Training Loop...
Epoch 1/3 | Training Focal Loss: 0.7071
Epoch 2/3 | Training Focal Loss: 0.6546
Epoch 3/3 | Training Focal Loss: 0.6486


## 4. Multi-Class Isotonic Calibration
Once the model is trained, we pass the Validation Set through it. 
We extract the raw softmax probabilities, and fit three separate `IsotonicRegression` models (one for Away, one for Draw, one for Home) against the true outcomes. 
Finally, we normalize them so they always sum perfectly to 1.0.

In [6]:
model.eval()
val_logits = []
val_targets = []

with torch.no_grad():
    for batch_cont, batch_league, batch_targets in val_loader:
        logits = model(batch_cont, batch_league)
        val_logits.append(logits)
        val_targets.append(batch_targets)
        
val_logits = torch.cat(val_logits, dim=0)
val_targets = torch.cat(val_targets, dim=0)

# Get raw softmax probabilities
raw_probs = F.softmax(val_logits, dim=1).numpy()
y_true = val_targets.numpy()

# Fit 3 independent Isotonic Regressors
iso_regressors = {}
classes = [0, 1, 2] # Away, Draw, Home
calibrated_probs = np.zeros_like(raw_probs)

for c in classes:
    iso = IsotonicRegression(out_of_bounds='clip')
    # Binary target: 1 if this class was the true outcome, 0 otherwise
    y_binary = (y_true == c).astype(int)
    
    # Fit the regressor on the raw probabilities for this class
    calibrated_probs[:, c] = iso.fit_transform(raw_probs[:, c], y_binary)
    iso_regressors[c] = iso

# Normalize so the 3 calibrated probabilities sum to 1.0 for every row
row_sums = calibrated_probs.sum(axis=1)[:, np.newaxis]
# Avoid divide-by-zero just in case
row_sums[row_sums == 0] = 1.0 
calibrated_probs_normalized = calibrated_probs / row_sums

# Show comparison on the first 5 samples
df_compare = pd.DataFrame({
    "True Outcome": y_true[:5],
    "Raw_Away": raw_probs[:5, 0], "Cal_Away": calibrated_probs_normalized[:5, 0],
    "Raw_Draw": raw_probs[:5, 1], "Cal_Draw": calibrated_probs_normalized[:5, 1],
    "Raw_Home": raw_probs[:5, 2], "Cal_Home": calibrated_probs_normalized[:5, 2],
})

print("Calibration Complete! Sample comparison:")
display(df_compare.round(3))

print("\n✅ The Training and Calibration loop is ready for real data.")


Calibration Complete! Sample comparison:


,True Outcome,Raw_Away,Cal_Away,Raw_Draw,Cal_Draw,Raw_Home,Cal_Home
0,1,0.295,0.323,0.390,0.345,0.314,0.332
1,1,0.293,0.323,0.386,0.345,0.321,0.332
2,1,0.297,0.323,0.402,0.345,0.301,0.332
3,0,0.299,0.323,0.368,0.345,0.333,0.332
4,0,0.301,0.323,0.400,0.345,0.299,0.332



✅ The Training and Calibration loop is ready for real data.
